# SAMV-FC: Slot-Attributed Multi-Task Learning for Fact-Checking
---

### Sections Overview

| Section | Description |
|---------|-------------|
| 1 | Environment & Dependencies |
| 2 | Experimental Configuration |
| 3 | Data Loader & Automated Conflict Slot Labeling |
| 4 | Dataset & Difference Markers |
| 5 | Intermediate Vietnamese NLI Fine-Tuning |
| 6 | SAMV-FC Architecture | 
| 7 | Multi-Task Optimization |
| 8 | Evaluation & Result |

### Notebook Setup 
1. Navigate to *Runtime → Change runtime type* and choose **GPU (T4, V100, or A100)**.
2. Verify your desired dataset in Section 2 (`DATASET_NAME = 'viwikifc'` or `'vifactcheck'`), then select *Runtime → Run all*.

## 1. Environment & Dependencies

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
ON_COLAB = True

# 2. Install dependencies
!pip install -q transformers datasets scikit-learn accelerate "kagglehub[pandas-datasets]"

# 3. Core libraries
import os, re, json, time, random, shutil
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# 4. Device verification
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Experimental Configuration

In [ ]:
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

DATASET_NAME = 'viwikifc'  # 'vifactcheck' | 'viwikifc'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KAGGLE_DATASET = 'thaonngyn/data-fc'
DATASET_FILES = {
    'vifactcheck': {
        'train': 'train-vifactcheck.csv',
        'dev': 'dev-vifactcheck.csv',
        'test': 'test-vifactcheck.csv',
    },
    'viwikifc': {
        'train': 'train-factcheckwiki.csv',
        'dev': 'dev-factcheckwiki.csv',
        'test': 'test-factcheckwiki.csv',
    },
}

# Ablations & Regularization
USE_NLI_INTERMEDIATE = True
NLI_SOURCE = 'xnli'
NLI_TRAIN_SAMPLES = 50000
NLI_EPOCHS = 2
FORCE_RETRAIN_NLI = False
NLI_LR = 1e-5
NLI_MAX_LEN = 128
NLI_BATCH_SIZE = 16
NLI_GRAD_ACCUM = 2
NLI_SEED = 42

USE_DIFF_MARKERS = True
USE_PAIR_POOLING = True
LABEL_SMOOTHING = 0.1
R_DROP_ALPHA = 1.0

RUN_TAG = '_'.join(tag for tag, on in [
    ('nli', USE_NLI_INTERMEDIATE),
    ('diff', USE_DIFF_MARKERS),
    ('pair', USE_PAIR_POOLING),
    (f'ls{LABEL_SMOOTHING:g}', LABEL_SMOOTHING > 0),
    (f'rdrop{R_DROP_ALPHA:g}', R_DROP_ALPHA > 0),
] if on) or 'base'

# Paths & Checkpoints
BASE_DIR = '/content/drive/MyDrive/NCKH' if os.path.exists('/content/drive/MyDrive') else os.path.abspath('./colab_working')
WORKING_DIR = os.path.join(BASE_DIR, 'main_runs')
OUT_DIR = os.path.join(WORKING_DIR, f'samv_fc_large_{DATASET_NAME}_{RUN_TAG}')
os.makedirs(OUT_DIR, exist_ok=True)

ENSEMBLE_SEEDS = [1, 10, 100, 1000, 10000]
EXISTING_CHECKPOINTS = {}
KEEP_CHECKPOINTS = True

# Hyperparameters
MODEL_KEY = 'xlm-roberta-large'
MAX_LEN = 256
BATCH_SIZE = 4 if R_DROP_ALPHA > 0 else 8
GRAD_ACCUM = 16 // BATCH_SIZE
MAX_GRAD_NORM = 1.0

LR_BACKBONE = 5e-6
LR_HEADS = 1e-4
HEAD_WARMUP_EPOCHS = 1
NUM_EPOCHS = 8
DROPOUT = 0.1
LAMBDA_CONFLICT = 0.35

## 3. Data Loader & Conflict Slot Labeling

In [ ]:
LABEL2ID = {'Supported': 0, 'Refuted': 1, 'NEI': 2}
ID2LABEL = {0: 'Supported', 1: 'Refuted', 2: 'NEI'}

CONFLICT_MAP = {'NONE': 0, 'TEMPORAL': 1, 'NUMERIC': 2, 'ENTITY': 3, 'NEGATION': 4}
ID2CONFLICT = {v: k for k, v in CONFLICT_MAP.items()}

def extract_years(text: str) -> List[int]:
    return sorted(set(int(y) for y in re.findall(r'\b((?:16|17|18|19|20)\d{2})\b', str(text))))

def extract_numbers(text: str) -> List[float]:
    nums = []
    for raw in re.findall(r'(?<!\w)(\d+(?:[.,]\d+)*)(?!\w)', str(text)):
        try:
            n = raw.replace(' ', '').replace('.', '').replace(',', '.')
            val = float(n)
            if val < 1600 or val > 2100:
                nums.append(val)
        except ValueError:
            pass
    return sorted(set(nums))

def detect_conflict_type(claim: str, evidence: str, label_str: str) -> str:
    if label_str != 'Refuted':
        return 'NONE'

    c_years, e_years = extract_years(claim), extract_years(evidence)
    if c_years and e_years and set(c_years) != set(e_years):
        return 'TEMPORAL'

    c_nums, e_nums = extract_numbers(claim), extract_numbers(evidence)
    if c_nums and e_nums and set(c_nums) != set(e_nums):
        return 'NUMERIC'

    negations = {'không', 'chưa', 'chẳng', 'không_thể', 'chưa_từng', 'phủ_nhận'}
    if any(w in claim.lower() for w in negations) != any(w in evidence.lower() for w in negations):
        return 'NEGATION'

    return 'ENTITY'

@dataclass
class MultiTaskSample:
    id: str
    claim: str
    evidence: str
    label_verdict: int
    label_conflict: int
    conflict_name: str

def find_dataset_file(fname: str) -> str:
    for d in ['/content/drive/MyDrive/NCKH/data', '/content/drive/MyDrive/data', '/content/data', './data', './dataset_raw']:
        p = Path(d)
        if p.exists():
            matches = list(p.rglob(fname))
            if matches:
                return str(matches[0])

    import kagglehub
    local_root = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    matches = list(local_root.rglob(fname))
    if matches:
        return str(matches[0])
    raise FileNotFoundError(f"File '{fname}' not found in Drive or Kagglehub cache.")

def load_split_df(dataset: str, split: str) -> pd.DataFrame:
    return pd.read_csv(find_dataset_file(DATASET_FILES[dataset][split]))

def load_multitask_data(dataset: str, split: str, limit: Optional[int] = None) -> List[MultiTaskSample]:
    df = load_split_df(dataset, split)
    if limit:
        df = df.head(limit)

    samples = []
    for idx, r in df.iterrows():
        if dataset == 'vifactcheck':
            sid = str(r.get('annotation_id') or idx)
            c = str(r.get('Statement') or '').strip()
            e = str(r.get('Evidence') or '').strip()
            raw_l = r.get('labels')
            l_map = {0: 'Supported', 1: 'Refuted', 2: 'NEI', '0': 'Supported', '1': 'Refuted', '2': 'NEI'}
        else:
            sid = str(r.get('pairID') or idx)
            c = str(r.get('claim') or '').strip()
            e = str(r.get('evidence') or '').strip()
            raw_l = r.get('gold_label')
            l_map = {'Supports': 'Supported', 'Refutes': 'Refuted', 'Not_Enough_Information': 'NEI'}

        l_str = l_map.get(raw_l)
        if not l_str or not c:
            continue

        c_type = detect_conflict_type(c, e, l_str)
        samples.append(MultiTaskSample(
            id=sid, claim=c, evidence=e,
            label_verdict=LABEL2ID[l_str],
            label_conflict=CONFLICT_MAP[c_type],
            conflict_name=c_type,
        ))

    print(f'Loaded {len(samples)} samples from {dataset} [{split}]')
    return samples

train_samples = load_multitask_data(DATASET_NAME, 'train')
dev_samples = load_multitask_data(DATASET_NAME, 'dev')
test_samples = load_multitask_data(DATASET_NAME, 'test')


## 4. PyTorch Dataset & Tokenizer


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_KEY)
DIFF_OPEN, DIFF_CLOSE = '<diff>', '</diff>'
if USE_DIFF_MARKERS:
    tokenizer.add_special_tokens({'additional_special_tokens': [DIFF_OPEN, DIFF_CLOSE]})

_EDGE_PUNCT = re.compile(r'^\W+|\W+$')

def mark_claim_diff(claim: str, evidence: str) -> str:
    ev_words = {_EDGE_PUNCT.sub('', w.lower()) for w in evidence.split()}
    out, span = [], False
    for w in claim.split():
        norm = _EDGE_PUNCT.sub('', w.lower())
        novel = bool(norm) and norm not in ev_words
        if novel and not span:
            out.append(DIFF_OPEN); span = True
        elif not novel and span:
            out.append(DIFF_CLOSE); span = False
        out.append(w)
    if span:
        out.append(DIFF_CLOSE)
    return ' '.join(out)

def resize_with_mean_init(model, vocab_size: int):
    old_size = model.get_input_embeddings().weight.shape[0]
    if old_size != vocab_size:
        model.resize_token_embeddings(vocab_size)
        with torch.no_grad():
            emb = model.get_input_embeddings().weight
            emb[old_size:] = emb[:old_size].mean(dim=0, keepdim=True)

def encode_pair(first: str, second: str, max_len: int) -> Dict[str, torch.Tensor]:
    if USE_DIFF_MARKERS:
        first = mark_claim_diff(first, second)
    enc = tokenizer(first, second, truncation=True, padding='max_length', max_length=max_len)
    seq = enc.sequence_ids()
    return {
        'input_ids': torch.tensor(enc['input_ids'], dtype=torch.long),
        'attention_mask': torch.tensor(enc['attention_mask'], dtype=torch.long),
        'claim_mask': torch.tensor([1 if s == 0 else 0 for s in seq], dtype=torch.long),
        'evidence_mask': torch.tensor([1 if s == 1 else 0 for s in seq], dtype=torch.long),
    }

MODEL_INPUT_KEYS = ('input_ids', 'attention_mask', 'claim_mask', 'evidence_mask')
def model_inputs(batch) -> Dict[str, torch.Tensor]:
    return {k: batch[k].to(DEVICE) for k in MODEL_INPUT_KEYS}

class MultiTaskFactDataset(Dataset):
    def __init__(self, samples: List[MultiTaskSample], max_len: int = 256):
        self.samples = samples
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        item = encode_pair(s.claim, s.evidence, self.max_len)
        item['label_verdict'] = torch.tensor(s.label_verdict, dtype=torch.long)
        item['label_conflict'] = torch.tensor(s.label_conflict, dtype=torch.long)
        return item

train_loader = DataLoader(MultiTaskFactDataset(train_samples, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(MultiTaskFactDataset(dev_samples, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(MultiTaskFactDataset(test_samples, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

## 5. Intermediate Vietnamese NLI Fine-Tuning

In [ ]:
from transformers import AutoModel

class SAMVFactClassifier(nn.Module):
    def __init__(self, model_name: str, num_verdict_labels: int = 3, num_conflict_labels: int = 5,
                 dropout: float = 0.1, pair_pooling: bool = True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.pair_pooling = pair_pooling

        if pair_pooling:
            self.pair_proj = nn.Sequential(
                nn.Linear(5 * hidden_size, hidden_size),
                nn.GELU(),
                nn.Dropout(dropout)
            )

        self.conflict_proj = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.conflict_head = nn.Linear(hidden_size // 2, num_conflict_labels)

        self.conflict_gate = nn.Sequential(
            nn.Linear(hidden_size // 2, hidden_size),
            nn.Sigmoid()
        )
        self.conflict_shift_proj = nn.Linear(hidden_size // 2, hidden_size)

        self.verdict_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_verdict_labels)
        )

    @staticmethod
    def _masked_mean(hidden, mask):
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1.0)

    def forward(self, input_ids, attention_mask, claim_mask=None, evidence_mask=None):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True).last_hidden_state
        cls_rep = hidden[:, 0, :]

        if self.pair_pooling:
            claim_rep = self._masked_mean(hidden, claim_mask)
            evidence_rep = self._masked_mean(hidden, evidence_mask)
            pair = torch.cat([cls_rep, claim_rep, evidence_rep,
                              (claim_rep - evidence_rep).abs(), claim_rep * evidence_rep], dim=-1)
            rep = self.pair_proj(self.dropout(pair))
        else:
            rep = self.dropout(cls_rep)

        conflict_feat = self.conflict_proj(rep)
        logits_conflict = self.conflict_head(conflict_feat)

        gate = self.conflict_gate(conflict_feat)
        conflict_shift = self.conflict_shift_proj(conflict_feat)
        fused_rep = rep + gate * conflict_shift
        logits_verdict = self.verdict_head(fused_rep)

        return logits_verdict, logits_conflict, gate

ConflictGatedFactClassifier = SAMVFactClassifier

def build_model() -> SAMVFactClassifier:
    model = SAMVFactClassifier(
        BACKBONE_PATH, num_verdict_labels=3, num_conflict_labels=5,
        dropout=DROPOUT, pair_pooling=USE_PAIR_POOLING
    )
    resize_with_mean_init(model.backbone, len(tokenizer))
    return model.to(DEVICE)

print(f'SAMV-FC ready | backbone: {BACKBONE_PATH} | pair_pooling: {USE_PAIR_POOLING} | diff_markers: {USE_DIFF_MARKERS}')

## 6. SAMV-FC Architecture

In [ ]:
from transformers import AutoModel

class SAMVFactClassifier(nn.Module):
    def __init__(self, model_name: str, num_verdict_labels: int = 3, num_conflict_labels: int = 5,
                 dropout: float = 0.1, pair_pooling: bool = True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.pair_pooling = pair_pooling

        if pair_pooling:
            self.pair_proj = nn.Sequential(
                nn.Linear(5 * hidden_size, hidden_size),
                nn.GELU(),
                nn.Dropout(dropout)
            )

        self.conflict_proj = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.conflict_head = nn.Linear(hidden_size // 2, num_conflict_labels)

        self.conflict_gate = nn.Sequential(
            nn.Linear(hidden_size // 2, hidden_size),
            nn.Sigmoid()
        )
        self.conflict_shift_proj = nn.Linear(hidden_size // 2, hidden_size)

        self.verdict_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_verdict_labels)
        )

    @staticmethod
    def _masked_mean(hidden, mask):
        weights = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1.0)

    def forward(self, input_ids, attention_mask, claim_mask=None, evidence_mask=None):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True).last_hidden_state
        cls_rep = hidden[:, 0, :]

        if self.pair_pooling:
            claim_rep = self._masked_mean(hidden, claim_mask)
            evidence_rep = self._masked_mean(hidden, evidence_mask)
            pair = torch.cat([cls_rep, claim_rep, evidence_rep,
                              (claim_rep - evidence_rep).abs(), claim_rep * evidence_rep], dim=-1)
            rep = self.pair_proj(self.dropout(pair))
        else:
            rep = self.dropout(cls_rep)

        conflict_feat = self.conflict_proj(rep)
        logits_conflict = self.conflict_head(conflict_feat)

        gate = self.conflict_gate(conflict_feat)
        conflict_shift = self.conflict_shift_proj(conflict_feat)
        fused_rep = rep + gate * conflict_shift
        logits_verdict = self.verdict_head(fused_rep)

        return logits_verdict, logits_conflict, gate

ConflictGatedFactClassifier = SAMVFactClassifier

def build_model() -> SAMVFactClassifier:
    model = SAMVFactClassifier(
        BACKBONE_PATH, num_verdict_labels=3, num_conflict_labels=5,
        dropout=DROPOUT, pair_pooling=USE_PAIR_POOLING
    )
    resize_with_mean_init(model.backbone, len(tokenizer))
    return model.to(DEVICE)

print(f'SAMV-FC ready | backbone: {BACKBONE_PATH} | pair_pooling: {USE_PAIR_POOLING} | diff_markers: {USE_DIFF_MARKERS}')

## 7. Multi-Task Optimization

In [ ]:
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup

class_weights = torch.tensor([1.0, 1.20, 1.0], device=DEVICE)
loss_fn_verdict = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
loss_fn_conflict = nn.CrossEntropyLoss()

DEV_V_TRUES = np.array([s.label_verdict for s in dev_samples])
DEV_C_TRUES = np.array([s.label_conflict for s in dev_samples])

def predict_probs(model, loader):
    model.eval()
    v_probs, c_probs = [], []
    with torch.no_grad():
        for batch in loader:
            logits_v, logits_c, _ = model(**model_inputs(batch))
            v_probs.append(torch.softmax(logits_v.float(), dim=-1).cpu().numpy())
            c_probs.append(torch.softmax(logits_c.float(), dim=-1).cpu().numpy())
    return np.concatenate(v_probs), np.concatenate(c_probs)

def refuted_conflict_acc(v_trues, c_trues, c_preds):
    idx = np.where(v_trues == 1)[0]
    return accuracy_score(c_trues[idx], c_preds[idx]) if len(idx) else 0.0

def train_one_seed(seed: int, ckpt_path: str):
    set_seed(seed)
    model = build_model()

    backbone_params = list(model.backbone.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith('backbone.')]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_BACKBONE, 'weight_decay': 1e-2},
        {'params': head_params, 'lr': LR_HEADS, 'weight_decay': 1e-3}
    ], eps=1e-8)

    effective_steps = (len(train_loader) // GRAD_ACCUM) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(effective_steps * 0.1), effective_steps)
    scaler = torch.cuda.amp.GradScaler()

    best_f1, best_epoch = -1.0, 0
    print(f'\n--- [Seed {seed}] Training SAMV-FC Large ---')
    print(f'Backbone LR: {LR_BACKBONE} | Heads LR: {LR_HEADS} | Warmup Epochs: {HEAD_WARMUP_EPOCHS} | Total: {NUM_EPOCHS}')

    for epoch in range(NUM_EPOCHS):
        backbone_trainable = epoch >= HEAD_WARMUP_EPOCHS
        for p in model.backbone.parameters():
            p.requires_grad = backbone_trainable

        model.train()
        total_loss_verdict = total_loss_consist = total_loss_kl = 0.0
        t0 = time.time()

        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            inputs = model_inputs(batch)
            labels_v = batch['label_verdict'].to(DEVICE)
            labels_c = batch['label_conflict'].to(DEVICE)

            with torch.cuda.amp.autocast():
                if R_DROP_ALPHA > 0:
                    doubled = {k: torch.cat([v, v], dim=0) for k, v in inputs.items()}
                    labels_v = torch.cat([labels_v, labels_v], dim=0)
                    labels_c = torch.cat([labels_c, labels_c], dim=0)
                    logits_v, logits_c, gate = model(**doubled)
                    logp_a, logp_b = F.log_softmax(logits_v.float(), dim=-1).chunk(2, dim=0)
                    l_kl = 0.5 * (F.kl_div(logp_a, logp_b, reduction='batchmean', log_target=True)
                                  + F.kl_div(logp_b, logp_a, reduction='batchmean', log_target=True))
                else:
                    logits_v, logits_c, gate = model(**inputs)
                    l_kl = torch.zeros((), device=DEVICE)

                l_v = loss_fn_verdict(logits_v, labels_v)
                l_c = loss_fn_conflict(logits_c, labels_c)

                prob_v = torch.softmax(logits_v, dim=-1)
                prob_c = torch.softmax(logits_c, dim=-1)
                l_consist = torch.mean(prob_v[:, 0] * (1.0 - prob_c[:, 0]))

                loss = (l_v + LAMBDA_CONFLICT * l_c + LAMBDA_CONSISTENCY * l_consist + R_DROP_ALPHA * l_kl) / GRAD_ACCUM

            scaler.scale(loss).backward()
            total_loss_verdict += l_v.item()
            total_loss_consist += l_consist.item()
            total_loss_kl += l_kl.item()

            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad()

        dev_v_probs, dev_c_probs = predict_probs(model, dev_loader)
        dev_v_preds = dev_v_probs.argmax(axis=1)
        dev_macro_f1 = f1_score(DEV_V_TRUES, dev_v_preds, average='macro')
        dev_acc = accuracy_score(DEV_V_TRUES, dev_v_preds)
        c_acc = refuted_conflict_acc(DEV_V_TRUES, DEV_C_TRUES, dev_c_probs.argmax(axis=1))

        phase_str = 'FULL' if backbone_trainable else 'WARMUP'
        print(f'[Seed {seed}] Epoch {epoch+1:02d}/{NUM_EPOCHS:02d} [{time.time()-t0:.0f}s] ({phase_str}) | '
              f'Loss(V): {total_loss_verdict/len(train_loader):.3f} | '
              f'Loss(Consist): {total_loss_consist/len(train_loader):.3f} | '
              f'Dev Macro-F1: {dev_macro_f1*100:.2f}% | Dev Acc: {dev_acc*100:.2f}% | '
              f'Conflict Acc: {c_acc*100:.1f}%')

        if dev_macro_f1 > best_f1:
            best_f1, best_epoch = dev_macro_f1, epoch + 1
            torch.save(model.state_dict(), ckpt_path)
            print(f'  --> [BEST] Saved {ckpt_path} (Dev F1: {dev_macro_f1*100:.2f}%)')

    return model, best_f1, best_epoch

def seed_probs_path(seed: int) -> str:
    return os.path.join(OUT_DIR, f'ensemble_probs_seed{seed}.npz')

for seed in ENSEMBLE_SEEDS:
    probs_path = seed_probs_path(seed)
    if os.path.exists(probs_path):
        print(f'[Seed {seed}] Found cached predictions at {probs_path} - skipping.')
        continue

    if seed in EXISTING_CHECKPOINTS:
        ckpt_path = EXISTING_CHECKPOINTS[seed]
        print(f'[Seed {seed}] Using existing checkpoint: {ckpt_path}')
        set_seed(seed)
        model = build_model()
        best_epoch = -1
    else:
        ckpt_path = os.path.join(OUT_DIR, f'best_multitask_xlmr_large_seed{seed}.pt')
        model, _, best_epoch = train_one_seed(seed, ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    dev_v_probs, dev_c_probs = predict_probs(model, dev_loader)
    test_v_probs, test_c_probs = predict_probs(model, test_loader)
    dev_macro_f1 = f1_score(DEV_V_TRUES, dev_v_probs.argmax(axis=1), average='macro')

    np.savez(
        probs_path,
        dev_verdict=dev_v_probs, dev_conflict=dev_c_probs,
        test_verdict=test_v_probs, test_conflict=test_c_probs,
        best_epoch=best_epoch, dev_macro_f1=dev_macro_f1, checkpoint_path=ckpt_path,
    )
    print(f'[Seed {seed}] Best epoch {best_epoch} | Dev Macro-F1: {dev_macro_f1*100:.2f}% | Saved: {probs_path}')

    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    if not KEEP_CHECKPOINTS and seed not in EXISTING_CHECKPOINTS and os.path.exists(ckpt_path):
        os.remove(ckpt_path)
        print(f'[Seed {seed}] Removed checkpoint to conserve storage.')

print('\nCompleted training and inference for all seeds:', ENSEMBLE_SEEDS)

## 8. Evaluation & Result

In [ ]:
test_labels = ['Supported', 'Refuted', 'NEI']
test_v_trues = np.array([s.label_verdict for s in test_samples])
test_c_trues = np.array([s.label_conflict for s in test_samples])

missing = [s for s in ENSEMBLE_SEEDS if not os.path.exists(seed_probs_path(s))]
if missing:
    raise RuntimeError(f'Missing prediction probabilities for seed {missing}. Please re-run training.')

per_seed, dev_stack, test_stack, test_c_stack = [], [], [], []
for seed in ENSEMBLE_SEEDS:
    d = np.load(seed_probs_path(seed))
    dev_stack.append(d['dev_verdict'])
    test_stack.append(d['test_verdict'])
    test_c_stack.append(d['test_conflict'])
    seed_preds = d['test_verdict'].argmax(axis=1)
    per_seed.append({
        'seed': seed,
        'best_epoch': int(d['best_epoch']),
        'checkpoint_path': str(d['checkpoint_path']),
        'dev_macro_f1': float(d['dev_macro_f1']),
        'test_macro_f1': float(f1_score(test_v_trues, seed_preds, average='macro')),
        'test_accuracy': float(accuracy_score(test_v_trues, seed_preds)),
    })

test_probs = np.mean(test_stack, axis=0)
test_c_probs = np.mean(test_c_stack, axis=0)
dev_probs = np.mean(dev_stack, axis=0)
test_v_preds = test_probs.argmax(axis=1)
test_c_preds = test_c_probs.argmax(axis=1)

best_dev_macro_f1 = f1_score(DEV_V_TRUES, dev_probs.argmax(axis=1), average='macro')
test_macro_f1 = f1_score(test_v_trues, test_v_preds, average='macro')
test_acc = accuracy_score(test_v_trues, test_v_preds)
report = classification_report(test_v_trues, test_v_preds, target_names=test_labels, output_dict=True, digits=4)
cm = confusion_matrix(test_v_trues, test_v_preds)
c_test_acc = refuted_conflict_acc(test_v_trues, test_c_trues, test_c_preds)

ddof = 1 if len(per_seed) > 1 else 0
single_f1 = np.array([r['test_macro_f1'] for r in per_seed])
single_acc = np.array([r['test_accuracy'] for r in per_seed])
single_f1_mean, single_f1_std = single_f1.mean(), single_f1.std(ddof=ddof)
single_acc_mean, single_acc_std = single_acc.mean(), single_acc.std(ddof=ddof)
dev_f1_mean = np.mean([r['dev_macro_f1'] for r in per_seed])

print('\n' + '='*65)
print(f'INDIVIDUAL SEED RESULTS ON {DATASET_NAME.upper()} TEST (n={len(test_samples)})')
print('='*65)
for r in per_seed:
    print(f"  seed {r['seed']:>5} | best epoch {r['best_epoch']:>2} | Dev F1 {r['dev_macro_f1']*100:.2f}% | "
          f"Test F1 {r['test_macro_f1']*100:.2f}% | Test Acc {r['test_accuracy']*100:.2f}%")
print(f'  1 seed (mean ± std): Macro-F1 {single_f1_mean*100:.2f} ± {single_f1_std*100:.2f}% | '
      f'Acc {single_acc_mean*100:.2f} ± {single_acc_std*100:.2f}%')

print('\n' + '='*65)
print(f'ENSEMBLE RESULTS ({len(ENSEMBLE_SEEDS)} SEEDS) - SAMV-FC LARGE (n={len(test_samples)})')
print('='*65)
print(f'Dev Macro-F1 : {best_dev_macro_f1*100:.2f}%')
print(f'Macro-F1     : {test_macro_f1*100:.2f}%')
print(f'Accuracy     : {test_acc*100:.2f}%')
for lab in test_labels:
    print(f'  {lab:10s} - P: {report[lab]["precision"]*100:.1f}% | R: {report[lab]["recall"]*100:.1f}% | F1: {report[lab]["f1-score"]*100:.1f}%')
print('='*65)
print(f'Conflict Slot Accuracy (Refuted Test subset): {c_test_acc*100:.2f}%')

pct = lambda x: round(float(x) * 100, 2)
metrics_result = {
    'dataset': DATASET_NAME,
    'model_key': MODEL_KEY,
    'num_test_samples': len(test_samples),
    'aggregation': 'mean_softmax',
    'ensemble_seeds': ENSEMBLE_SEEDS,
    'macro_f1': pct(test_macro_f1),
    'accuracy': pct(test_acc),
    'conflict_slot_acc': pct(c_test_acc),
    'best_dev_macro_f1': pct(best_dev_macro_f1),
    'single_seed': {
        'macro_f1_mean': pct(single_f1_mean),
        'macro_f1_std': pct(single_f1_std),
        'accuracy_mean': pct(single_acc_mean),
        'accuracy_std': pct(single_acc_std),
        'dev_macro_f1_mean': pct(dev_f1_mean),
        'runs': [
            {**r, 'dev_macro_f1': pct(r['dev_macro_f1']),
             'test_macro_f1': pct(r['test_macro_f1']), 'test_accuracy': pct(r['test_accuracy'])}
            for r in per_seed
        ],
    },
    'per_class': {
        lab: {
            'precision': pct(report[lab]['precision']),
            'recall': pct(report[lab]['recall']),
            'f1_score': pct(report[lab]['f1-score']),
            'support': int(report[lab]['support']),
        }
        for lab in test_labels
    },
    'confusion_matrix': cm.tolist(),
    'hyperparameters': {
        'max_len': MAX_LEN,
        'batch_size': BATCH_SIZE,
        'grad_accum': GRAD_ACCUM,
        'effective_batch_size': BATCH_SIZE * GRAD_ACCUM,
        'lr_backbone': LR_BACKBONE,
        'lr_heads': LR_HEADS,
        'head_warmup_epochs': HEAD_WARMUP_EPOCHS,
        'epochs': NUM_EPOCHS,
        'dropout': DROPOUT,
        'lambda_conflict': LAMBDA_CONFLICT,
        'lambda_consistency': LAMBDA_CONSISTENCY,
        'run_tag': RUN_TAG,
        'backbone_path': BACKBONE_PATH,
        'nli_intermediate': USE_NLI_INTERMEDIATE,
        'diff_markers': USE_DIFF_MARKERS,
        'pair_pooling': USE_PAIR_POOLING,
        'label_smoothing': LABEL_SMOOTHING,
        'r_drop_alpha': R_DROP_ALPHA,
    },
    'saved_at': datetime.now(timezone.utc).isoformat(),
}

metrics_json_path = os.path.join(OUT_DIR, 'test_metrics_ensemble.json')
with open(metrics_json_path, 'w', encoding='utf-8') as f_m:
    json.dump(metrics_result, f_m, indent=2, ensure_ascii=False)

seed_test_preds = {seed: stack.argmax(axis=1) for seed, stack in zip(ENSEMBLE_SEEDS, test_stack)}
predictions_data = []
for i, s in enumerate(test_samples):
    pred_v_id, gold_v_id = int(test_v_preds[i]), int(test_v_trues[i])
    pred_c_id, gold_c_id = int(test_c_preds[i]), int(test_c_trues[i])
    probs = test_probs[i]
    predictions_data.append({
        'id': getattr(s, 'id', str(i)),
        'claim': s.claim,
        'evidence': s.evidence,
        'gold_verdict': ID2LABEL[gold_v_id],
        'pred_verdict': ID2LABEL[pred_v_id],
        'gold_verdict_id': gold_v_id,
        'pred_verdict_id': pred_v_id,
        'is_verdict_correct': bool(gold_v_id == pred_v_id),
        'verdict_probabilities': {
            'Supported': round(float(probs[0]), 4),
            'Refuted': round(float(probs[1]), 4),
            'NEI': round(float(probs[2]), 4),
        },
        'per_seed_pred_verdict': {str(seed): ID2LABEL[int(p[i])] for seed, p in seed_test_preds.items()},
        'gold_conflict': ID2CONFLICT.get(gold_c_id, str(gold_c_id)),
        'pred_conflict': ID2CONFLICT.get(pred_c_id, str(pred_c_id)),
        'gold_conflict_id': gold_c_id,
        'pred_conflict_id': pred_c_id,
        'is_conflict_correct': bool(gold_c_id == pred_c_id),
    })

predictions_json_path = os.path.join(OUT_DIR, 'test_predictions_ensemble.json')
with open(predictions_json_path, 'w', encoding='utf-8') as f_p:
    json.dump(predictions_data, f_p, indent=2, ensure_ascii=False)

print(f'\nSaved Test Metrics to    : {metrics_json_path}')
print(f'Saved Test Predictions to: {predictions_json_path} ({len(predictions_data)} samples)\n')

for f_name in sorted(os.listdir(OUT_DIR)):
    f_path = os.path.join(OUT_DIR, f_name)
    if os.path.isfile(f_path):
        size_kb = os.path.getsize(f_path) / 1024
        size_str = f'{size_kb/1024:.2f} MB' if size_kb > 1024 else f'{size_kb:.1f} KB'
        print(f'  [Artifact] {f_name:32s} ({size_str})')